# E4/E5 UK Data-Centre Demand Scaling & Projections

**Date:** 2025-08-12

Reconstruct absolute kW/kWh from UKPN % traces via a single-server anchor, classify sites, validate medians vs implied servers, scale to UK totals, and project 5–10y.


In [1]:

# %pip install pandas pyarrow numpy pyyaml scikit-learn matplotlib
import os, json, warnings
from typing import List, Optional, Dict, Tuple
import numpy as np, pandas as pd, yaml
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pathlib import Path

CLEAN_DIR  = Path (r"D:/CIM Warwick/Dissertation/Datasets/All Datasets/clean/")
os.makedirs("outputs", exist_ok=True)

PATH_UKPN = os.path.join(CLEAN_DIR,"ukpn_timeseries_tidy.parquet")
PATH_WS_A = os.path.join(CLEAN_DIR,"workstation_2021_may_aug_tidy.parquet")
PATH_WS_B = os.path.join(CLEAN_DIR,"workstation_2021_aug_dec_tidy.parquet")
PATH_COUNTS = os.path.join(CLEAN_DIR,"global_dc_counts_tidy.parquet")

UKPN_SITE_COL, UKPN_TIME_COL = "site_id","datetime_local"
PCT_COLS = ["util_pct","percent_of_peak","pct_peak","load_pct"]
KW_COLS, KVAR_COLS, KVA_COLS = ["active_power_kw","electricity_kW","kW"], ["reactive_power_kvar","kVAr"], ["apparent_power_kva","kVA"]
MIC_COLS = ["max_import_capacity_kva","mic_kva","MIC_kVA","mpan_mic_kva"]
PF_DEFAULT = 0.95

TYPE_RULES = [("edge_enterprise",0.0,3.0),("retail_colo",3.0,10.0),("wholesale_colo",10.0,50.0),("hyperscale",50.0,float("inf"))]
SERVER_W = 450.0
CONCURRENCY = 1.0

def pick_col(df, cands):
    for c in cands:
        if c in df.columns: return c
    return None

def servers_per_MW(server_W): return 1_000_000.0/server_W

def classify_type_by_MW(mw):
    for name,lo,hi in TYPE_RULES:
        if (mw>=lo) and (mw<hi): return name
    return "unknown"


## Single-server anchor

In [2]:

def ws_anchor(paths):
    frames=[pd.read_parquet(p) for p in paths if os.path.exists(p)]
    if not frames: raise FileNotFoundError("Workstation parquet(s) missing.")
    ws=pd.concat(frames,ignore_index=True)
    if "power_w" not in ws.columns: raise KeyError("Expected 'power_w' in workstation data.")
    return {"srv_kw_p95":float(ws.power_w.quantile(0.95)/1000),
            "srv_kw_p99":float(ws.power_w.quantile(0.99)/1000),
            "srv_kw_max":float(ws.power_w.max()/1000)}

server_anchor=ws_anchor([PATH_WS_A,PATH_WS_B])
server_kw_peak=server_anchor["srv_kw_p99"]
server_anchor


{'srv_kw_p95': 0.1146, 'srv_kw_p99': 0.1316, 'srv_kw_max': 0.22}

## Load UKPN & obtain % of peak

In [3]:
def load_ukpn():
    import numpy as np
    import pandas as pd

    df = pd.read_parquet(PATH_UKPN)

    # sanity-check required columns
    missing = [c for c in (UKPN_SITE_COL, UKPN_TIME_COL) if c not in df.columns]
    if missing:
        raise KeyError(f"UKPN dataset missing required columns: {missing}; available: {list(df.columns)}")

    # Use your 'utilisation' column as % of peak (accepts 0–1 or 0–100)
    if "utilisation" not in df.columns:
        raise KeyError("Expected 'utilisation' column in UKPN dataset but did not find it.")

    # Coerce to numeric and normalise: if values look like 0..1, convert to 0..100
    util = pd.to_numeric(df["utilisation"], errors="coerce")
    if util.max() is not None and util.max() <= 1.2 and util.max() > 0.01:
        util = util * 100.0

    df["util_pct_auto"] = util
    df["__pct_colname"] = "util_pct_auto"

    # Standardise time
    df[UKPN_TIME_COL] = pd.to_datetime(df[UKPN_TIME_COL])
    return df

# Re-run the loader
ukpn = load_ukpn()
print("Detected % column:", ukpn["__pct_colname"].iloc[0], "| Sites:", ukpn[UKPN_SITE_COL].nunique(), "| Rows:", len(ukpn))
ukpn.head(3)

Detected % column: util_pct_auto | Sites: 83 | Rows: 1580697


,site_id,site_type,voltage_level,datetime_local,utilisation,util_pct_auto,__pct_colname
0,Data Centre #55,Co-located,High Voltage Import,2024-05-24 23:30:00+00:00,0.2795,27.95,util_pct_auto
1,Data Centre #70,Co-located,Low Voltage Import,2024-05-24 23:30:00+00:00,0.2710,27.10,util_pct_auto
2,Data Centre #56,Co-located,High Voltage Import,2024-05-24 23:30:00+00:00,0.1400,14.00,util_pct_auto


## MIC, MW proxy, and type classification

In [4]:

def extract_mic_mw(df, pf=PF_DEFAULT):
    mic=pick_col(df,MIC_COLS)
    meta=df[[UKPN_SITE_COL]].drop_duplicates().copy()
    meta["mic_kva"]=np.nan
    if mic is not None:
        meta=meta.merge(df.groupby(UKPN_SITE_COL)[mic].max().rename("mic_kva"),
                        left_on=UKPN_SITE_COL,right_index=True,how="left")
    meta["mw_capacity_proxy"]= (meta["mic_kva"]*pf/1000.0) if "mic_kva" in meta else np.nan
    meta["site_type"]=meta["mw_capacity_proxy"].apply(lambda x: classify_type_by_MW(x) if pd.notnull(x) else "unknown")
    return meta

def classify_by_shape(df, n_clusters=3):
    pct=df["__pct_colname"].iloc[0]
    tmp=df.copy(); tmp["date"]=tmp[UKPN_TIME_COL].dt.date
    daily=tmp.groupby([UKPN_SITE_COL,"date"])[pct].agg(["mean","max"]).reset_index()
    feats=daily.groupby(UKPN_SITE_COL).agg(mean_pct=("mean","mean"),max_pct=("max","mean"),var_pct=("mean","var")).fillna(0).reset_index()
    km=KMeans(n_clusters=n_clusters,random_state=42,n_init="auto").fit(feats[["mean_pct","max_pct","var_pct"]].values)
    feats["shape_cluster"]=km.labels_
    order=feats.groupby("shape_cluster")["max_pct"].mean().sort_values().index.tolist()
    mapping={cl:i for i,cl in enumerate(order)}
    feats["site_type"]=feats["shape_cluster"].map({mapping.get(k):v for v,k in enumerate(order)})
    # map 0,1,2 -> edge, retail, wholesale
    feats["site_type"]=feats["shape_cluster"].map({order[0]:"edge_enterprise", order[1]:"retail_colo", order[2]:"wholesale_colo"})
    return feats[[UKPN_SITE_COL,"site_type","shape_cluster"]]

meta=extract_mic_mw(ukpn)
if meta["site_type"].eq("unknown").all():
    print("No MIC found; using shape-based classification.")
    meta=classify_by_shape(ukpn)
meta.head(5)


No MIC found; using shape-based classification.


,site_id,site_type,shape_cluster
0,Data Centre #1,edge_enterprise,0
1,Data Centre #10,edge_enterprise,0
2,Data Centre #11,retail_colo,1
3,Data Centre #12,wholesale_colo,2
4,Data Centre #13,edge_enterprise,0


## Reconstruct absolute series from % (server anchor → site peak, cap by MIC if available)

In [5]:

def default_servers_per_site_from_MW(server_W=SERVER_W):
    median_MW={"edge_enterprise":1.5,"retail_colo":5.0,"wholesale_colo":30.0,"hyperscale":75.0}
    spmw=servers_per_MW(server_W) # ~2222 servers/MW at 450W
    return {k:int(v*spmw) for k,v in median_MW.items()}

def reconstruct_absolute(df_ukpn, site_meta, server_kw_peak, concurrency, servers_per_site):
    df=df_ukpn.merge(site_meta,on=UKPN_SITE_COL,how="left")
    pct=df["__pct_colname"].iloc[0]
    n_srv=df["site_type"].map(lambda t: servers_per_site.get(t, servers_per_site.get("edge_enterprise",3000)))
    df["peak_kw_model"]=n_srv*server_kw_peak*concurrency
    if "mw_capacity_proxy" in df.columns:
        df["mic_cap_kw"]=df["mw_capacity_proxy"]*1000.0
        df["peak_kw"]=np.where(df["mic_cap_kw"].notna(), np.minimum(df["peak_kw_model"], df["mic_cap_kw"]), df["peak_kw_model"])
    else:
        df["peak_kw"]=df["peak_kw_model"]
    df["kw"]=df["peak_kw"]*(df[pct]/100.0)
    df["kwh"]=df["kw"]*0.5
    return df

servers_per_site=default_servers_per_site_from_MW()
df_abs=reconstruct_absolute(ukpn, meta, server_kw_peak, CONCURRENCY, servers_per_site)
df_abs.head(2)


KeyError: 'site_type'

## Validate medians vs implied servers at peak

In [ ]:

def implied_servers_per_site(df_abs, server_kw_peak, concurrency):
    peaks=df_abs.groupby(UKPN_SITE_COL)["kw"].max().rename("observed_peak_kw")
    meta=df_abs[[UKPN_SITE_COL,"site_type","peak_kw"]].drop_duplicates(UKPN_SITE_COL)
    meta=meta.merge(peaks,left_on=UKPN_SITE_COL,right_index=True,how="left")
    meta["implied_servers_at_peak"]=(meta["observed_peak_kw"]/(server_kw_peak*concurrency)).clip(lower=0)
    return meta

implied=implied_servers_per_site(df_abs, server_kw_peak, CONCURRENCY)
implied.to_csv("outputs/ukpn_implied_servers_per_site.csv", index=False)
implied.describe()


## Scale sample → UK totals via Cloudscene count

In [ ]:

def scale_to_uk_counts(df_abs, counts_path, country_name="United Kingdom"):
    if not os.path.exists(counts_path):
        warnings.warn("Counts parquet missing; returning sample totals.")
        nat=df_abs.groupby([UKPN_TIME_COL]).agg(kw=("kw","sum"),kwh=("kwh","sum")).reset_index()
        return df_abs.assign(kw_nat=df_abs["kw"],kwh_nat=df_abs["kwh"]), nat
    counts=pd.read_parquet(counts_path)
    uk_total=int(counts.loc[counts["country"]==country_name,"dc_count"].max())
    st=df_abs[[UKPN_SITE_COL,"site_type"]].drop_duplicates()
    type_counts=st["site_type"].value_counts().to_dict(); total=len(st)
    split={t: (c/total) for t,c in type_counts.items()}
    nat_counts={t: max(1,int(round(uk_total*share))) for t,share in split.items()}
    st["scale_factor"]=st["site_type"].map(lambda t: nat_counts.get(t,0)/type_counts.get(t,1))
    sf_map=st.set_index(UKPN_SITE_COL)["scale_factor"]
    out=df_abs.copy(); out["scale_factor"]=out[UKPN_SITE_COL].map(sf_map)
    out["kw_nat"]=out["kw"]*out["scale_factor"]; out["kwh_nat"]=out["kwh"]*out["scale_factor"]
    nat=out.groupby([UKPN_TIME_COL]).agg(kw=("kw_nat","sum"),kwh=("kwh_nat","sum")).reset_index()
    return out, nat

df_abs_nat, uk_nat = scale_to_uk_counts(df_abs, PATH_COUNTS, "United Kingdom")
uk_nat.to_csv("outputs/uk_nat_timeseries_halfhour.csv", index=False)
uk_nat.head(2)


## Scenario projections (annual TWh by type)

In [ ]:

scenario_cfg={"base_year":2024,"horizon_year":2035,
              "types":{"hyperscale":{"annual_growth":0.20,"mw_site_creep":0.03},
                       "wholesale_colo":{"annual_growth":0.13,"mw_site_creep":0.02},
                       "retail_colo":{"annual_growth":0.13,"mw_site_creep":0.01},
                       "edge_enterprise":{"annual_growth":0.03,"mw_site_creep":0.00}}}

def run_projection(df_abs_nat, cfg):
    base_year=int(cfg["base_year"]); horizon=int(cfg["horizon_year"])
    df=df_abs_nat.copy(); df["year"]=df[UKPN_TIME_COL].dt.year; df["month"]=df[UKPN_TIME_COL].dt.to_period("M")
    base=df[df["year"]==base_year]
    monthly=base.groupby(["month","site_type"]).agg(kwh=("kwh_nat","sum")).reset_index()
    out=[]
    for t,params in cfg["types"].items():
        g=params.get("annual_growth",0.0); m=params.get("mw_site_creep",0.0)
        for y in range(base_year,horizon+1):
            n=y-base_year; mult=(1+g)**n * (1+m)**n
            mt=monthly[monthly["site_type"]==t].copy()
            if mt.empty: continue
            mt["year"]=y; mt["kwh_proj"]=mt["kwh"]*mult; out.append(mt[["month","site_type","year","kwh_proj"]])
    if not out: return pd.DataFrame()
    proj=pd.concat(out,ignore_index=True)
    return proj.groupby(["year","site_type"])["kwh_proj"].sum().rename("kwh_year").reset_index()

proj=run_projection(df_abs_nat, scenario_cfg)
proj.to_csv("outputs/uk_dc_projection_annual.csv", index=False)
proj.head(6)


## Quick plots

In [ ]:

# Load-duration curve
sorted_kw=np.sort(uk_nat["kw"].values)[::-1]
x=np.arange(1,len(sorted_kw)+1)/len(sorted_kw)*100.0
plt.figure(); plt.plot(x, sorted_kw); plt.xlabel("% of half-hour slots"); plt.ylabel("kW (national)"); plt.title("UK Load-Duration Curve"); plt.grid(True,linestyle="--",alpha=0.4); plt.show()

# Annual TWh stacked by type
if not proj.empty:
    pvt=proj.pivot(index="year",columns="site_type",values="kwh_year").fillna(0)/1e6
    pvt.plot(kind="area"); plt.xlabel("Year"); plt.ylabel("TWh"); plt.title("Projection (annual)"); plt.show()


## Save a small summary JSON

In [ ]:

summary={"server_anchor_kw":server_anchor,
         "servers_per_site_defaults": default_servers_per_site_from_MW(),
         "sites_in_sample": int(df_abs["site_id"].nunique()),
         "rows_reconstructed": int(len(df_abs)),
         "national_rows": int(len(uk_nat)),
         "projection_rows": int(len(proj))}
with open("outputs/summary.json","w") as f: json.dump(summary,f,indent=2)
summary
